In [66]:
import sys
from pathlib import Path

project_root = Path.cwd().parent
sys.path.insert(0, str(project_root))
import utils, prompts

In [97]:
import os
import re
import json
import ast
import importlib

import pandas as pd

In [3]:
from dotenv import load_dotenv
load_dotenv()

True

In [4]:
gigachat_ak = os.getenv("GIGACHAT_AK")
persist_dir = '../db/chroma'

In [5]:
LLM = "GigaChat-2"

In [11]:
EMBED_MODEL = "Qwen/Qwen3-Embedding-0.6B"
#EMBED_MODEL = "sentence-transformers/all-MiniLM-L6-v2"

### Запрос пользователя для прогона по пайплайну

In [7]:
user_query = "Посоветуй ЖК бизнесс класса не дальше 1.5 км от метро"

In [8]:
right_answer = "['Верейская 41', 'Тропарево Парк', 'Октябрьская 98']"

### Подключение к векторной базе

In [9]:
import chromadb
from langchain_huggingface import HuggingFaceEmbeddings
from langchain_chroma import Chroma

In [10]:
client = chromadb.PersistentClient(path=persist_dir)
collections = client.list_collections()
collections

[Collection(name=chunk_700_qwen),
 Collection(name=chunk_300_se),
 Collection(name=chunk_500_qwen),
 Collection(name=chunk_300_qwen),
 Collection(name=chunk_700_se),
 Collection(name=chunk_500_se),
 Collection(name=chunk_900_se),
 Collection(name=chunk_900_qwen)]

In [12]:
collection_name = "chunk_700_qwen"

In [14]:
embeddings = HuggingFaceEmbeddings(model_name=EMBED_MODEL)

In [15]:
vectorstore = Chroma(
            persist_directory=persist_dir,
            embedding_function=embeddings,
            collection_name=collection_name,
        )

### Выделение ключей из запроса

In [16]:
from langchain_core.messages import SystemMessage, HumanMessage
from langchain_core.prompts import ChatPromptTemplate
from langchain_gigachat.chat_models import GigaChat
from langchain_core.runnables import RunnablePassthrough
from langchain_core.output_parsers import JsonOutputParser

In [17]:
json_parser = JsonOutputParser()

In [18]:
metadata_prompt = ChatPromptTemplate.from_messages([
    ("system", prompts.metadata_system),
    ("human", "{{question}}"),
], template_format="jinja2")

In [19]:
llm = GigaChat(
    credentials=gigachat_ak,
    model=LLM,  
    verify_ssl_certs=False 
)

In [20]:
extractor_chain = (
    RunnablePassthrough()
    | metadata_prompt 
    | llm 
    | json_parser
)

### Пример выделения метаданных

In [21]:
response = extractor_chain.invoke({"question": user_query})

In [22]:
response

{'h_class': 'Бизнес', 'dist_to_metro': {'$lte': 1.5}}

In [23]:
parsed_filter = utils.parse_filter(response)
parsed_filter

{'h_class': 'Бизнес', 'dist_to_metro': {'$lte': 1.5}}

In [24]:
normalized_filter = utils.normalize_chroma_filter(parsed_filter)
normalized_filter

{'$and': [{'h_class': {'$eq': 'Бизнес'}}, {'dist_to_metro': {'$lte': 1.5}}]}

### Готовим ретривер BM25

In [26]:
from langchain_community.retrievers import BM25Retriever
from langchain_core.documents import Document
from collections import defaultdict

In [27]:
all_docs = vectorstore.get()

In [28]:
meta = all_docs["metadatas"]
docs = all_docs["documents"]

In [29]:
docs_for_bm25 = [
    Document(page_content=d, metadata=m)
    for d, m in zip(docs, meta)
]

In [30]:
bm25_retriever = BM25Retriever.from_documents(docs_for_bm25, k=10)

In [31]:
bm25_retriever_with_lema = BM25Retriever.from_documents(docs_for_bm25, 
                                                        preprocess_func=utils.russian_preprocess,
                                                        k=10)

### Готовим векторный ретривер

In [32]:
search_kwargs = {"k": 10}
if normalized_filter is not None:
    search_kwargs["filter"] = normalized_filter
    print(f"[RAG] Поиск с фильтром {normalized_filter}")
else:
    print("[RAG] Фильтр не извлечён или пуст → поиск без фильтра")
vector_retriever = vectorstore.as_retriever(search_kwargs=search_kwargs)
docs = vector_retriever.invoke(user_query)
print(f"[RAG] Найдено документов векторным поиском: {len(docs)}")
if normalized_filter and not docs:
    print("[RAG] Поиск с фильтром по запросу вернул пустоту → повторный поиск без фильтра")
    search_kwargs = {"k": 10}
    vector_retriever = vectorstore.as_retriever(search_kwargs=search_kwargs)
    docs = vector_retriever.invoke(user_query)
    print(f"[RAG] Повторный поиск без фильтра → найдено: {len(docs)}")

[RAG] Поиск с фильтром {'$and': [{'h_class': {'$eq': 'Бизнес'}}, {'dist_to_metro': {'$lte': 1.5}}]}
[RAG] Найдено документов векторным поиском: 8


### Гибридный поиск

In [33]:
#retrievers = [bm25_retriever, vector_retriever]
retrievers_lema = [bm25_retriever_with_lema, vector_retriever]

In [34]:
#docs = utils.ensemble_retrieve(user_query, retrievers, weights=[0.5, 0.5], k=5)
docs = utils.ensemble_retrieve(user_query, retrievers_lema, weights=[0.5, 0.5], k=5)

### Augmented: обогащение контекстом

In [36]:
test_context = utils.get_context(docs)
test_context

'[START_BLOCK][START_META_BLOCK]Застройщик: MR Group\nНазвание ЖК: МЫС\nКласс жилья: Комфорт\nРайон: Северное Тушино\nМетро: Планерная\nРасстояние до метро в км: 1.2\nКоличество жильцов: 1500\nПланируемое количество мест в школе: 0\nДоступных мест в школе сейчас: 0\nПланируемое количество мест в детсадах: 0\nДоступных мест в детсадах сейчас: 0\nТип парковки: Подземный\n[END_META_BLOCK]МЫС: резидентов будут радовать цветистые сады размером парка зарядье благоустроенная набережная качелями пруда безопасные игровые площадки детей возрастов лаунжпространства фонтанами теплые беседки очага территории создана обширная спортивная инфраструктура километровый лыжный маршрут 10 км пешеходных 5 км велосипедных дорожек 4 теннисных корта крытый тренажерный зал йогабеседки 4 паделкорта 6 площадок сквоша современный скейтпарк кроме жители жк мыс смогут любое время посещать кафе видом пруд рестораны гастромаркете салоны красоты супермаркеты другие сервисы повседневной инфраструктуры которые расположат

### Generation: создание ответа

In [101]:
importlib.reload(prompts)
base_prompt = ChatPromptTemplate.from_messages([
    ('system', prompts.system_prompt_2),
    ('human', prompts.human_prompt)
], template_format="jinja2")

In [38]:
messages = base_prompt.format_messages(context=test_context, question=user_query)

In [39]:
response = llm.invoke(messages)

In [40]:
answer = response.content
answer

'Из представленных вариантов ЖК бизнес-класса, не дальше 1.5 км от метро, подходит:\n\n1. Октябрьская 98 (расстояние до метро: 0.7 км)\n\nОтвет: Октябрьская 98'

### Функция rag_with_filter для пайплайна

In [41]:
def rag_with_filter(user_question: str) -> str:
    print(user_question)
    # 1. Получаем фильтр из вопроса
    raw_filter = extractor_chain.invoke({"question": user_question})
    print("DEBUG raw filter from LLM:", raw_filter, "type:", type(raw_filter))

    # 2. Парсим фильтр в словарь
    parsed_filter = utils.parse_filter(raw_filter)
    print("DEBUG parsed filter from LLM:", parsed_filter, "type:", type(parsed_filter))
            
    # 3. Нормализуем фильтр под требования Chroma
    normalized_filter = utils.normalize_chroma_filter(parsed_filter)
    print("DEBUG normalized filter:", normalized_filter, "type:", type(normalized_filter))

    # 4. Создаём ретривер для векторного поиска
    search_kwargs = {"k": 10}
    if normalized_filter is not None:
        search_kwargs["filter"] = normalized_filter
        print(f"[RAG] Поиск с фильтром {normalized_filter}")
    else:
        print("[RAG] Фильтр не извлечён или пуст → поиск без фильтра")
    
    vector_retriever = vectorstore.as_retriever(search_kwargs=search_kwargs)
    docs = vector_retriever.invoke(user_question)
    print(f"[RAG] Найдено документов векторным поиском: {len(docs)}")

    # 5. Если ретривер векторного поиска с фильтром ничего не вернул, то проводим повторный поиск без фильтра
    if normalized_filter and not docs:
        print("[RAG] Поиск с фильтром по запросу вернул пустоту → повторный поиск без фильтра")
        search_kwargs = {"k": 10}
        vector_retriever = vectorstore.as_retriever(search_kwargs=search_kwargs)
        docs = vector_retriever.invoke(user_question)
        print(f"[RAG] Повторный поиск без фильтра → найдено: {len(docs)}")

    # 6. Получение ответов от ретриверов
    retrievers = [bm25_retriever, vector_retriever]
    docs = utils.ensemble_retrieve(user_question, retrievers, weights=None, k=5)
    
    # 7. Если и после этого пусто — не отправляем в LLM
    if not docs:
        return "Ответ: Нет данных", "Ответ: Нет данных"

    # 8. Формируем контекст
    context_str = utils.get_context(docs)

    # 9. Готовим промпт и вызываем LLM
    messages = base_prompt.format_messages(context=context_str, question=user_question)
    print(base_prompt)
    response = utils.safe_llm_invoke(llm, messages)
    print(response)

    return response.content, context_str

In [57]:
answer = rag_with_filter(user_query)

Посоветуй ЖК бизнесс класса не дальше 1.5 км от метро
DEBUG raw filter from LLM: {'h_class': 'Бизнес', 'dist_to_metro': {'$lte': 1.5}} type: <class 'dict'>
DEBUG parsed filter from LLM: {'h_class': 'Бизнес', 'dist_to_metro': {'$lte': 1.5}} type: <class 'dict'>
DEBUG normalized filter: {'$and': [{'h_class': {'$eq': 'Бизнес'}}, {'dist_to_metro': {'$lte': 1.5}}]} type: <class 'dict'>
[RAG] Поиск с фильтром {'$and': [{'h_class': {'$eq': 'Бизнес'}}, {'dist_to_metro': {'$lte': 1.5}}]}
[RAG] Найдено документов векторным поиском: 8
input_variables=['context', 'question'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='\nТы — эксперт по городской аналитике и недвижимости Москвы. Твоя задача — анализировать данные о жилых комплексах и подбирать варианты, соответствующие вопросу клиента.\n\nПРАВИЛА РАБОТЫ С КОНТЕКСТОМ:\n1. Используй ТОЛЬКО данные из переданного контекста. Ничего не 

In [58]:
print(answer[0])

Из представленных вариантов ЖК бизнес-класса, не дальше 1.5 км от метро подходят:

- Октябрьская 98 (расстояние до метро 0.7 км)
- Тропарево Парк (расстояние до метро 1.5 км)

Ответ: Октябрьская 98, Тропарево Парк


In [59]:
print(answer[1])

[START_BLOCK][START_META_BLOCK]Застройщик: Самолет
Название ЖК: Прибрежный Парк
Класс жилья: Комфорт
Район: Красногорск
Метро: Домодедовская
Расстояние до метро в км: 17.0
Количество жильцов: 8000
Планируемое количество мест в школе: 1650
Доступных мест в школе сейчас: 1650
Планируемое количество мест в детсадах: 450
Доступных мест в детсадах сейчас: 450
Тип парковки: Надземный
[END_META_BLOCK]Прибрежный Парк: берегу реки пахры 9 км мкад расположен жилой комплекс прибрежный парк река огибает квартал трех сторон рядом находятся видновский парк музейзаповедник горки ленинские москвы удобно добираться автомобиле общественном транспорте расположение 15 минутах пешком находится жд станция ленинская метро домодедовская около 25 минут автомобиле территории квартала расположены остановки общественного транспорта работают сервисы каршеринга кикшеринга автомобилистов предусмотрен собственный выезд каширское шоссе мкад 9 км аэропорта домодедово около 20 минут автомобиле окружение пахра становится

### Оценка качества на тестовом вопросе

In [98]:
importlib.reload(prompts)
metrics_prompt = ChatPromptTemplate.from_messages([
    ('system', prompts.sys_metrics_prompt_2),
    ('human', prompts.hum_metrics_prompt)
], template_format="jinja2")

In [61]:
answer = utils.truncate_before_answer(answer[0])
answer

' Октябрьская 98, Тропарево Парк'

In [62]:
answers = [{'answer': right_answer, 'answer_rag': answer}]

In [63]:
answers

[{'answer': "['Верейская 41', 'Тропарево Парк', 'Октябрьская 98']",
  'answer_rag': ' Октябрьская 98, Тропарево Парк'}]

In [99]:
metrics_message = metrics_prompt.format_messages(answers=str(answers))

In [100]:
metrics_message

[SystemMessage(content='\nТы — сверщик по ключевым объектам, сравни два текста и найди, какие сущности из эталонного answer есть в фактическом answer_rag. \n\nЗАДАЧА:\nДля каждого объекта в списке сравни верный ответ (answer) с ответом модели (answer_rag). Рассчитай метрику correct — это число от 0 до 1, отражающее точность воспроизведения фактов.\n\nПРАВИЛА ОЦЕНКИ:\n1. Если answer_rag содержит «Не знаю/Нет данных» или аналоги — correct = 0.\n2. Если в answer ровно один факт/значение, то:\n   - correct = 1, если answer_rag точно передаёт этот факт (допускаются незначительные перефразы без потери смысла);\n   - correct = 0, если есть значительные искажения, пропуски или ошибки.\n3. Если в answer — список допустимых вариантов (например, список ЖК, застройщиков, характеристик), то:\n   - Из answer_rag извлеки все упомянутые сущности (игнорируй пояснительный текст).\n   - Если в answer_rag есть хоть один элемент, которого нет в answer (ложное утверждение) — correct = 0.\n   - Если все элем

In [102]:
metrics_response = utils.safe_llm_invoke(llm, metrics_message)

In [103]:
text = metrics_response.content

In [104]:
data = ast.literal_eval(text)
data

[{'id': 0, 'correct': 0.5}]

### Оценка качества на ВСЕХ тестовых вопросах

In [105]:
questions = []

In [106]:
with open(r'../raw_data/questions_lite.jsonl', 'r', encoding='utf-8') as f:
    for line in f:
        record = json.loads(line)
        data = dict(record)
        questions.append(data)

In [107]:
questions

[{'question': 'Какое плановое количество жителей указано для проекта «Восточное Бутово»?',
  'answer': ['30000']},
 {'question': 'Относится ли проект «2-й Иртышский» к типу «Комфорт плюс»?',
  'answer': ['Да']},
 {'question': 'В каком проекте плановая вместимость школ больше: «Ильинские луга» или «Восточное Бутово»?',
  'answer': ['Восточное Бутово']},
 {'question': 'Какой проект расположен дальше от метро: «Квартал Авиаторов» или «Остафьево»?',
  'answer': ['Квартал Авиаторов']},
 {'question': 'В каком проекте плановое количество жителей больше: «Пригород Лесное» или «Новое Видное»?',
  'answer': ['Пригород Лесное']}]

In [108]:
for num, question in enumerate(questions):
    print(question)
    user_query = question['question']
    answer_rag, docs = rag_with_filter(user_query)
    question['id'] = num
    question['answer_rag'] = utils.truncate_before_answer(answer_rag)
    question['chunks'] = docs

{'question': 'Какое плановое количество жителей указано для проекта «Восточное Бутово»?', 'answer': ['30000']}
Какое плановое количество жителей указано для проекта «Восточное Бутово»?
DEBUG raw filter from LLM: {'proj': 'Восточное Бутово', 'num_resid': {'$exists': True}} type: <class 'dict'>
DEBUG parsed filter from LLM: {'proj': 'Восточное Бутово', 'num_resid': {'$exists': True}} type: <class 'dict'>
DEBUG normalized filter: {'proj': {'$eq': 'Восточное Бутово'}} type: <class 'dict'>
[RAG] Поиск с фильтром {'proj': {'$eq': 'Восточное Бутово'}}
[RAG] Найдено документов векторным поиском: 3
input_variables=['context', 'question'] input_types={} partial_variables={} messages=[SystemMessagePromptTemplate(prompt=PromptTemplate(input_variables=[], input_types={}, partial_variables={}, template='\nТы — эксперт по городской аналитике и недвижимости Москвы. Твоя задача — анализировать данные о жилых комплексах и подбирать варианты, соответствующие вопросу клиента.\n\nПРАВИЛА РАБОТЫ С КОНТЕКСТО

In [109]:
answers = [{'answer': el['answer'], 'answer_rag': el['answer_rag']} for el in questions]

In [110]:
metrics_message = metrics_prompt.format_messages(answers=str(answers))

In [111]:
metrics_response = utils.safe_llm_invoke(llm, metrics_message)

In [112]:
text = metrics_response.content

In [113]:
data = ast.literal_eval(text)

In [114]:
metrics_df = pd.DataFrame(data)
metrics_df.head()

,id,correct
0,0,1
1,1,1
2,2,0
3,3,1
4,4,0


In [115]:
accuracy = metrics_df['correct'].sum() / len(metrics_df['correct'])
accuracy 

0.6